# SHAP por dentro — módulo 05, internals

Companheiro técnico do walkthrough. Quatro medições sobre o TreeSHAP
nativo do XGBoost:

1. **exatidão e determinismo** — aditividade no teste inteiro e três
   refits com contribuições idênticas;
2. **o base value** — de onde toda explicação parte;
3. **interações** — a matriz 41×41 soma de volta às contribuições;
4. **o custo do interventional** — erro de Monte Carlo × número de
   permutações, contra o exato.

In [1]:
%pip install -q xgboost scikit-learn scipy matplotlib numpy pandas pyarrow

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pathlib
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xgboost as xgblib
from scipy.special import expit

pd.set_option("display.width", 120)
RANDOM_STATE = 42

REPO = pathlib.Path.cwd()
while REPO != REPO.parent and not (REPO / "tools" / "srag_model.py").exists():
    REPO = REPO.parent
if not (REPO / "tools" / "srag_model.py").exists():
    import tempfile
    REPO = pathlib.Path(tempfile.gettempdir()) / "interpretable-ml-lectures"
    if not REPO.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/wbendinelli/interpretable-ml-lectures.git",
                        str(REPO)], check=True)
sys.path.insert(0, str(REPO / "tools"))
import srag_model as M

FIGDIR = pathlib.Path("figures_generated")
FIGDIR.mkdir(exist_ok=True)

g = M.load_gold(REPO / "modules/00-dataset/gold/gold_covid_obito_sample.parquet")
xgb, _logit = M.fit_models(g)
paciente = M.pick_exemplar(xgb, g)
booster = xgb.get_booster()


def dmatrix(frame):
    x, _, _ = frame_x = M.design_matrix(frame)
    return xgblib.DMatrix(x, enable_categorical=True)


teste = g[g["split"] == "test"].reset_index(drop=True)
dm_teste = dmatrix(teste)
contrib = booster.predict(dm_teste, pred_contribs=True)
margem = booster.predict(dm_teste, output_margin=True)
NOMES = list(M.FEATURES) + ["base"]

desvio = float(np.abs(contrib.sum(axis=1) - margem).max())
print(f"contribuições no teste: {contrib.shape[0]:,} × {contrib.shape[1]} "
      f"(40 features + base)".replace(",", "."))
print(f"aditividade exata: |soma das contribuições − margem| máximo = {desvio:.2e}")
print(f"base value (margem): {contrib[0, -1]:.4f} — "
      f"sigmoide dá {expit(contrib[0, -1]):.4f}")

contribuições no teste: 16.142 × 41 (40 features + base)
aditividade exata: |soma das contribuições − margem| máximo = 7.63e-06
base value (margem): -0.7919 — sigmoide dá 0.3118


## §1 — Exato e determinístico

Duas propriedades que nenhum método amostrado do curso tem, medidas.

In [3]:
dev_total = float(np.abs(contrib.sum(axis=1) - margem).max())
print(f"aditividade no teste inteiro ({len(contrib):,} linhas): "
      f"desvio máximo {dev_total:.2e}".replace(",", "."))

refits = []
for r in range(3):
    m_r, _ = M.fit_models(g)
    c_r = m_r.get_booster().predict(dm_teste, pred_contribs=True)
    refits.append(c_r)
d01 = float(np.abs(refits[0] - refits[1]).max())
d02 = float(np.abs(refits[0] - refits[2]).max())
print(f"3 refits, mesmas contribuições: desvio máx {max(d01, d02):.2e}")
print("=> determinístico como o módulo 01 mediu para o modelo; a explicação")
print("   herda o determinismo — zero variância de semente, zero amostragem.")

aditividade no teste inteiro (16.142 linhas): desvio máximo 7.63e-06


3 refits, mesmas contribuições: desvio máx 0.00e+00
=> determinístico como o módulo 01 mediu para o modelo; a explicação
   herda o determinismo — zero variância de semente, zero amostragem.


## §2 — O base value

`pred_contribs` devolve 41 colunas; a última é o ponto de partida.

In [4]:
x_tr, _, y_tr = M.design_matrix(g[g["split"] == "train"])
m_tr = booster.predict(xgblib.DMatrix(x_tr, enable_categorical=True),
                       output_margin=True)
base = float(contrib[0, -1])
print(f"base value: {base:+.4f} (margem)")
print(f"margem média do treino: {m_tr.mean():+.4f}")
print(f"sigmoide(base) = {expit(base):.4f} | taxa de óbito do treino: "
      f"{y_tr.mean():.4f}")
print()
print("o base é o ponto de partida de toda explicação — a predição de um")
print("paciente sobre o qual nada se sabe — e mora perto da prevalência do")
print("treino; a distância que sobra é o que o boosting moveu do chute inicial.")

base value: -0.7919 (margem)
margem média do treino: -1.0177
sigmoide(base) = 0.3118 | taxa de óbito do treino: 0.3136

o base é o ponto de partida de toda explicação — a predição de um
paciente sobre o qual nada se sabe — e mora perto da prevalência do
treino; a distância que sobra é o que o boosting moveu do chute inicial.


## §3 — As interações somam de volta

`pred_interactions` decompõe cada contribuição em efeitos principais +
pares. A verificação de consistência é a soma; o subproduto é saber qual
par o modelo mais usa em conjunto.

In [5]:
sub = np.arange(2000)
inter = booster.predict(xgblib.DMatrix(M.design_matrix(teste.iloc[sub])[0],
                                       enable_categorical=True),
                        pred_interactions=True)
print(f"matriz de interações: {inter.shape} (2.000 linhas × 41 × 41)")
soma_linhas = inter.sum(axis=2)
dev_i = float(np.abs(soma_linhas - contrib[sub]).max())
print(f"soma das interações de cada feature == contribuição: desvio máx {dev_i:.2e}")
par = np.abs(inter[:, :-1, :-1]).mean(axis=0)
np.fill_diagonal(par, 0.0)
i, j = np.unravel_index(np.argmax(par), par.shape)
print(f"o par que mais interage no teste: {NOMES[i]} × {NOMES[j]} "
      f"(média|φ_ij| = {par[i, j]:.4f})")

matriz de interações: (2000, 41, 41) (2.000 linhas × 41 × 41)
soma das interações de cada feature == contribuição: desvio máx 7.87e-06
o par que mais interage no teste: idade_anos × meses_desde_mar2020 (média|φ_ij| = 0.1914)


## §4 — O custo do interventional

O walkthrough §4 estimou Shapley por permutação com 60 amostras. Aqui, o
erro dessa estimativa em função do orçamento, contra o exato de caminho —
lembrando que os dois não convergem um para o outro: respondem
condicionais diferentes.

In [6]:
train_pool = g[g["split"] == "train"].reset_index(drop=True)
fundo = train_pool.sample(200, random_state=RANDOM_STATE).reset_index(drop=True)


def phi_idade(n_perm, semente):
    rng = np.random.default_rng(semente)
    acc = 0.0
    for t in range(n_perm):
        bg = fundo.iloc[int(rng.integers(len(fundo)))]
        ordem_f = list(rng.permutation(M.FEATURES))
        linha = {c: bg[c] for c in g.columns}
        linhas = [dict(linha)]
        for c in ordem_f:
            linha[c] = paciente[c]
            linhas.append(dict(linha))
        frame = pd.DataFrame(linhas).astype(g.dtypes)
        m = booster.predict(dmatrix(frame), output_margin=True)
        k = ordem_f.index("idade_anos")
        acc += float(m[k + 1] - m[k]) / n_perm
    return acc


exato = float(contrib[teste.index[teste["gold_id"] == int(paciente["gold_id"])][0],
                      NOMES.index("idade_anos")])
print(f"pred_contribs de idade (exato, caminho): {exato:+.4f}")
print("permutações   5 estimativas (sementes 0–4)                 dp")
medias = {}
for n in (10, 30, 60):
    ests = [phi_idade(n, s) for s in range(5)]
    medias[n] = float(np.mean(ests))
    print(f"{n:11d}   " + " ".join(f"{e:+.3f}" for e in ests)
          + f"   {np.std(ests):.4f}")
print()
print(f"com 5 repetições o dp ainda oscila — o decaimento √n aparece devagar.")
print(f"E repare no CENTRO: a média com 60 permutações é {medias[60]:+.3f}, perto")
print(f"do interventional do walkthrough §4, ACIMA do exato de caminho "
      f"({exato:+.4f}):")
print("as estimativas não convergem para o pred_contribs — medem outra")
print("condicional. O que não oscila: o exato custa UMA predição, dp zero.")

pred_contribs de idade (exato, caminho): +0.6689
permutações   5 estimativas (sementes 0–4)                 dp


         10   +0.495 +0.575 +0.797 +1.012 +0.716   0.1804


         30   +0.637 +0.442 +1.040 +0.988 +0.651   0.2271


         60   +0.650 +0.703 +1.006 +0.883 +0.778   0.1278

com 5 repetições o dp ainda oscila — o decaimento √n aparece devagar.
E repare no CENTRO: a média com 60 permutações é +0.804, perto
do interventional do walkthrough §4, ACIMA do exato de caminho (+0.6689):
as estimativas não convergem para o pred_contribs — medem outra
condicional. O que não oscila: o exato custa UMA predição, dp zero.


## Fechamento

TreeSHAP exato: uma predição por explicação, zero sementes, aditividade
em ponto flutuante. O que ele não decide por você: qual condicional
contar (caminho × intervenção) — e essa escolha, como todas as deste
curso, se mede: em contradições fabricadas por coalizão, pela mesma
`gate_impossible` de sempre.